# Fire Season Timing - Turkey Ecoregion Scale

In [ ]:
'''
----------------------------------------------------------------------------------------------------
Computes fire season timing metrics (onset, peak, end, season length)
for all WWF RESOLVE ecoregions intersecting Turkey, for years 2003-2024.

Data sources:
- MODIS Terra active fire: MODIS/061/MOD14A1
- MODIS Aqua active fire:  MODIS/061/MYD14A1
- Ecoregions:              RESOLVE/ECOREGIONS/2017
- Country boundary:        USDOS/LSIB_SIMPLE/2017

Output:
- outputs/turkey_ecoregions/<ECO_ID>_<ECO_NAME>.csv  (per ecoregion)
- outputs/turkey_ecoregions/master_turkey.csv        (combined)
'''

import ee
import pandas as pd
import matplotlib.pyplot as plt
import os
import time

In [ ]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Initialize(project='fire-seasons')

In [ ]:
# PARAMETERS ---------------------------------------------------------------------------------------

FIRE_MASK_MIN   = 8     # FireMask threshold: >= 8 = nominal + high confidence only
ONSET_THRESHOLD = 0.05  # Cumulative fraction threshold for fire season onset (5%)
END_THRESHOLD   = 0.95  # Cumulative fraction threshold for fire season end (95%)
MIN_DETECTIONS  = 20    # Minimum annual fire detections required to compute metrics
YEARS           = list(range(2003, 2026))  # Full study period: 2003–2025

In [ ]:
# LOAD MODIS COLLECTIONS ---------------------------------------------------------------------------
# Terra and Aqua are loaded once here at module level.
# Per-year and per-day filtering is handled inside get_daily_counts().

terra = ee.ImageCollection("MODIS/061/MOD14A1").select('FireMask')
aqua  = ee.ImageCollection("MODIS/061/MYD14A1").select('FireMask')

print('Terra image count:', terra.size().getInfo())
print('Aqua image count:', aqua.size().getInfo())
print('Terra and Aqua collections loaded.')

In [ ]:
# LOAD TURKEY ECOREGIONS ---------------------------------------------------------------------------

# Turkey boundary from LSIB
turkey = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
           .filter(ee.Filter.eq('country_na', 'Turkey'))

# RESOLVE ecoregions clipped to Turkey
ecoregions_turkey = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017").filterBounds(turkey.geometry())

# Inspect
n_eco    = ecoregions_turkey.size().getInfo()
eco_list = ecoregions_turkey.select(['ECO_ID', 'ECO_NAME']).getInfo()

print(f'Number of ecoregions intersecting Turkey: {n_eco}')
print()
for f in eco_list['features']:
    print(f['properties']['ECO_ID'], '|', f['properties']['ECO_NAME'])

In [ ]:
# BUILD ECOREGION RECORD LIST ----------------------------------------------------------------------

# Converts the GEE FeatureCollection into a plain Python list of dicts
# so we can iterate over ecoregions without repeated GEE calls.

eco_records = []
for f in eco_list['features']:
    eco_records.append({
        'eco_id'   : f['properties']['ECO_ID'],
        'eco_name' : f['properties']['ECO_NAME'],
        'geometry' : ee.Geometry(f['geometry'])
    })

print(f'Loaded {len(eco_records)} ecoregion records.')

## Helper Functions

In [ ]:
# FUNCTION: get_daily_counts -----------------------------------------------------------------------


def get_daily_counts(eco_geometry, year):
    """
    Compute daily MODIS active fire detection counts for a given
    ecoregion geometry and calendar year.

    Combines Terra (MOD14A1) and Aqua (MYD14A1) by taking the pixel-wise
    maximum across sensors for each day, deduplicating detections that
    appear in both sensors on the same day.

    All 365 daily counts are retrieved in a SINGLE reduceRegion call
    by stacking all daily images into one multi-band image using toBands().
    This avoids the 'Too many concurrent aggregations' error that occurs
    when reduceRegion is called inside a mapped function.

    Parameters
    ----------
    eco_geometry : ee.Geometry
        The geometry of the ecoregion to compute counts for.
    year : int
        The calendar year to process (e.g. 2008).

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
          - doy           : int, day of year (1-indexed)
          - n_detections  : int, number of fire pixels detected
        One row per day of the year (365 or 366 rows).
    """

    import calendar

    start  = ee.Date.fromYMD(year, 1, 1)
    end    = ee.Date.fromYMD(year + 1, 1, 1)
    n_days = 366 if calendar.isleap(year) else 365

    # Pre-filter both collections to this year
    terra_year = terra.filterDate(start, end)
    aqua_year  = aqua.filterDate(start, end)

    # Fallback empty image for days where a sensor returns no image
    empty = ee.Image.constant(0).rename('FireMask').toUint8()

    # Server-side list of day offsets: [0, 1, 2, ... n_days-1]
    day_seq = ee.List.sequence(0, n_days - 1)

    def make_daily_image(d):
        """
        For a single day offset d, build a deduplicated binary fire image.
        Returns a single-band image named by its DOY (e.g. 'day_001').
        No reduceRegion here — reduction happens once outside this function.
        """
        d        = ee.Number(d)
        date     = start.advance(d, 'day')
        date_end = date.advance(1, 'day')

        terra_day = terra_year.filterDate(date, date_end)
        aqua_day  = aqua_year.filterDate(date, date_end)

        # Use empty fallback if sensor has no image for this day
        t = ee.Image(ee.Algorithms.If(
            terra_day.size().gt(0),
            terra_day.select('FireMask').max(),
            empty
        ))
        a = ee.Image(ee.Algorithms.If(
            aqua_day.size().gt(0),
            aqua_day.select('FireMask').max(),
            empty
        ))

        # Pixel-wise max across sensors = deduplication
        combined    = t.max(a)
        fire_binary = combined.gte(FIRE_MASK_MIN).unmask(0)

        # Name this band by its DOY so we can identify it after toBands()
        # ee.Number.format creates a string like '001', '002', ... '365'
        band_name = ee.String('day_').cat(
            d.add(1).toInt().format('%03d')
        )

        return fire_binary.rename(band_name)

    # Build a collection of 365 single-band images
    daily_collection = ee.ImageCollection(day_seq.map(make_daily_image))

    # Stack all 365 bands into ONE multi-band image
    # This is the key change — instead of 365 separate images,
    # we now have one image with 365 bands (one per day)
    stacked = daily_collection.toBands()

    # ONE single reduceRegion call on the entire stacked image
    # This counts fire pixels for all 365 days in a single aggregation
    counts_dict = stacked.reduceRegion(
        reducer   = ee.Reducer.sum(),
        geometry  = eco_geometry,
        scale     = 1000,
        maxPixels = 1e9
    ).getInfo()  # one single getInfo() call — brings all 365 counts at once

    # counts_dict looks like: {'0_day_001': 5, '1_day_002': 0, ...}
    # Parse it back into a tidy DataFrame
    rows = []
    for band_name, count in sorted(counts_dict.items()):
        # Extract the DOY number from the band name (e.g. '0_day_001' → 1)
        doy = int(band_name.split('_')[-1])
        rows.append({
            'doy'         : doy,
            'n_detections': int(count) if count is not None else 0
        })

    return pd.DataFrame(rows)

In [ ]:
# FUNCTION: compute_timing_metrics -----------------------------------------------------------------

def compute_timing_metrics(df, year):
    """
    Compute fire season timing metrics from a daily detection count DataFrame.

    Onset and end are defined by percentile thresholds on the cumulative
    detection fraction (5% and 95% respectively). Peak is defined as the
    fire activity centroid — the detection-weighted mean DOY — which is
    more robust to sparse outlier detections than the rolling-mean maximum
    and is guaranteed to fall within the onset–end window.

    Returns None if total detections fall below MIN_DETECTIONS, or if
    onset/end cannot be computed.

    Parameters
    ----------
    df : pd.DataFrame
        Daily counts DataFrame with columns 'doy' and 'n_detections',
        as returned by get_daily_counts().
    year : int
        The calendar year being processed (used for logging only).

    Returns
    -------
    dict or None
        Dict with keys: year, onset_doy, peak_doy, end_doy,
        season_length, n_detections.
        Returns None if metrics cannot be computed.
    """

    total = df['n_detections'].sum()

    if total < MIN_DETECTIONS:
        print(f'  {year}: insufficient detections ({total}), skipping.')
        return None

    df = df.copy().sort_values('doy')
    cumulative = df['n_detections'].cumsum()
    cum_frac   = cumulative / total

    # Onset: first DOY where cumulative fraction reaches 5%
    onset_rows = df[cum_frac >= ONSET_THRESHOLD]
    # End: first DOY where cumulative fraction reaches 95%
    end_rows   = df[cum_frac >= END_THRESHOLD]

    if onset_rows.empty or end_rows.empty:
        print(f'  {year}: could not compute onset or end, skipping.')
        return None

    onset_doy = int(onset_rows.iloc[0]['doy'])
    end_doy   = int(end_rows.iloc[0]['doy'])

    # Peak: fire activity centroid (detection-weighted mean DOY)
    # More robust than rolling-mean maximum; always falls within onset–end window
    weights   = df['n_detections']
    peak_doy  = int(round((df['doy'] * weights).sum() / weights.sum()))

    # Validate peak is within onset–end window
    if not (onset_doy <= peak_doy <= end_doy):
        print(f'  {year}: WARNING — peak ({peak_doy}) outside onset–end window '
              f'({onset_doy}–{end_doy}), flagging.')

    # Season length inclusive of both endpoints
    season_length = end_doy - onset_doy + 1

    return {
        'year'         : year,
        'onset_doy'    : onset_doy,
        'peak_doy'     : peak_doy,
        'end_doy'      : end_doy,
        'season_length': season_length,
        'n_detections' : int(total)
    }

## Main Pipeline

In [ ]:
# FULL PIPELINE LOOP — ALL ECOREGIONS x ALL YEARS --------------------------------------------------

# Saves a per-ecoregion CSV after each ecoregion completes so progress
# is not lost if the run is interrupted.

output_dir = 'outputs/turkey_ecoregions'
os.makedirs(output_dir, exist_ok=True)

all_metrics = []

for eco in eco_records:
    eco_id   = eco['eco_id']
    eco_name = eco['eco_name']
    geometry = eco['geometry']

    print(f'\n=== {eco_name} (ID: {eco_id}) ===')
    eco_metrics = []

    for year in YEARS:
        t0 = time.time()

        try:
            df_year = get_daily_counts(geometry, year)
            metrics = compute_timing_metrics(df_year, year)

            if metrics is not None:
                metrics['eco_id']   = eco_id
                metrics['eco_name'] = eco_name
                eco_metrics.append(metrics)
                all_metrics.append(metrics)

        except Exception as e:
            print(f'  {year}: ERROR — {e}')
            continue

        t1 = time.time()
        print(f'  {year}: done in {t1 - t0:.1f}s')

    # Save per-ecoregion CSV immediately after finishing all years
    if eco_metrics:
        eco_df    = pd.DataFrame(eco_metrics)
        safe_name = eco_name.replace(' ', '_').replace('/', '_')
        eco_path  = f'{output_dir}/{eco_id}_{safe_name}.csv'
        eco_df.to_csv(eco_path, index=False)
        print(f'  Saved {len(eco_metrics)} years → {os.path.abspath(eco_path)}')
    else:
        print(f'  No valid years for {eco_name}, skipping CSV.')

print('\nAll ecoregions complete.')

In [ ]:
# COMBINE ALL RESULTS INTO MASTER CSV --------------------------------------------------------------

master_df = pd.DataFrame(all_metrics)

# Reorder columns
master_df = master_df[['eco_id', 'eco_name', 'year',
                        'onset_doy', 'peak_doy', 'end_doy',
                        'season_length', 'n_detections']]

master_path = f'{output_dir}/master_turkey.csv'
master_df.to_csv(master_path, index=False)

print(f'Master CSV saved: {master_df.shape[0]} ecoregion-year rows.')
print(f'Path: {os.path.abspath(master_path)}')
print()
print(master_df.head(10))